<a href="https://colab.research.google.com/github/ksuaray/M4DS/blob/MATH-170-Spring-2026/MATH170_Final_Project_Beta_2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MATH 170: Calculus for Data Science**

# **Final Project Proof of Concept — Beta Distribution $(\alpha=2,\beta=5)$**
## **Marketing & Consumer Behavior: Customer Action Probability**

This notebook is a **worked example** of the final project structure.

It is organized in a paired format:
- a **Prompt / Problems** cell
- then a **Complete Solution** cell

The model in this notebook is:
$$
 f(x)=30x(1-x)^4,\qquad 0\le x\le 1
$$
with application to **customer action probability** such as clicking an ad or making a purchase.

We will use:
- **SymPy** for symbolic calculus
- **NumPy** for arrays and numerical work
- **Pandas** for simple data tables
- **Plotly** for graphs


## **Deliverables for the actual project**

For the real project, submit:
1. a **video report**
2. a Jupyter notebook
3. graphs and calculations

In this proof of concept, the notebook itself contains the worked example that could support such a video report.


---
## **Part 0 — Setup**
Run this cell first.
---


In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
import plotly.express as px
import plotly.graph_objects as go

sp.init_printing()

# Symbols
x, t = sp.symbols('x t', real=True)

# Beta(2,5) pdf on [0,1]
f = 30*x*(1-x)**4

print("Setup complete.")
print("pdf f(x) =", sp.expand(f))


In [ ]:

def sample_points(a, b, n, method='L'):
    dx = (b - a) / n
    if method == 'L':
        xs = a + np.arange(n) * dx
    elif method == 'R':
        xs = a + np.arange(1, n + 1) * dx
    elif method == 'M':
        xs = a + (np.arange(n) + 0.5) * dx
    else:
        raise ValueError("method must be 'L', 'R', or 'M'")
    return xs, dx


def riemann_sum(f_num, a, b, n, method='L'):
    xs, dx = sample_points(a, b, n, method)
    return np.sum(f_num(xs)) * dx


def riemann_table(f_num, a, b, n, method='L'):
    xs, dx = sample_points(a, b, n, method)
    vals = f_num(xs)
    left_edges = a + np.arange(n) * dx
    right_edges = a + np.arange(1, n + 1) * dx
    return pd.DataFrame({
        'subinterval': [f'[{left_edges[i]:.3g}, {right_edges[i]:.3g}]' for i in range(n)],
        'sample_x': xs,
        'height': vals,
        'rect_area': vals * dx
    })


def exact_integral(expr, a, b):
    return sp.integrate(expr, (x, a, b))


def plot_riemann(f_num, a, b, n=4, method='L', title='Riemann Sum'):
    xs_curve = np.linspace(a, b, 600)
    ys_curve = f_num(xs_curve)
    xs, dx = sample_points(a, b, n, method)
    heights = f_num(xs)
    left_edges = a + np.arange(n) * dx

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=xs_curve, y=ys_curve,
        mode='lines', name='curve',
        line=dict(width=3)
    ))

    for i in range(n):
        x0 = left_edges[i]
        x1 = x0 + dx
        h = float(heights[i])
        fig.add_shape(
            type='rect',
            x0=x0, x1=x1, y0=0, y1=h,
            line=dict(width=1),
            fillcolor='rgba(99, 110, 250, 0.28)'
        )
        fig.add_trace(go.Scatter(
            x=[xs[i]], y=[h],
            mode='markers',
            marker=dict(size=8),
            name='sample point' if i == 0 else None,
            showlegend=(i == 0)
        ))

    fig.add_hline(y=0, line_width=1)
    fig.update_layout(
        title=f'{title}: method={method}, n={n}',
        xaxis_title='x',
        yaxis_title='y',
        template='plotly_white',
        width=850,
        height=450
    )
    return fig


def convergence_table(f_num, expr, a, b, n_values, methods=('L','R','M')):
    exact_val = float(sp.N(exact_integral(expr, a, b)))
    rows = []
    for n in n_values:
        row = {'n': n, 'exact': exact_val}
        for method in methods:
            approx = riemann_sum(f_num, a, b, n, method)
            row[f'{method}_approx'] = approx
            row[f'{method}_error'] = approx - exact_val
        rows.append(row)
    return pd.DataFrame(rows)


---
## **Part 1 — Context and Data Analysis**
### **Prompt / Problems**

Let
$$
X = \text{probability that a customer takes an action}
$$
where the action could be clicking an ad, opening an email, or making a purchase.

1. Define $X$ in words.
2. State why $0\le X\le 1$.
3. Go to **Kaggle** and find a dataset related to customer behavior, conversion, marketing, ads, or purchases.
4. Identify or construct a variable representing a **probability** or **proportion**.
5. Extract at least 20 values between 0 and 1.
6. Display a table and a graph of the values.
7. Compute the minimum, maximum, and mean.

### Guiding Questions
- Why must this variable lie between 0 and 1?
- Does the data appear skewed or symmetric?
- Where are most values concentrated?


### **Complete Solution**

For this proof of concept, use a small example dataset representing customer purchase probabilities.

Interpretation:
- $X$ = the probability that a customer who sees a marketing campaign completes the target action.
- Since probabilities cannot be negative and cannot exceed 1, the valid interval is $[0,1]$.

Example data (30 values) are chosen to reflect low-to-moderate action probabilities, which is consistent with a right-skewed Beta$(2,5)$ shape.


In [ ]:
# Example probability data for the proof of concept
prob_data = np.array([
    0.03, 0.05, 0.06, 0.07, 0.08, 0.09,
    0.10, 0.11, 0.12, 0.13, 0.14, 0.15,
    0.16, 0.18, 0.19, 0.20, 0.21, 0.22,
    0.24, 0.25, 0.27, 0.29, 0.31, 0.33,
    0.36, 0.39, 0.42, 0.46, 0.51, 0.58
], dtype=float)

df = pd.DataFrame({"customer_action_probability": prob_data})
df.head(10)


In [ ]:
summary = pd.DataFrame({
    "statistic": ["min", "max", "mean"],
    "value": [prob_data.min(), prob_data.max(), prob_data.mean()]
})
summary


In [ ]:
fig = px.histogram(
    df,
    x="customer_action_probability",
    nbins=10,
    title="Example Customer Action Probabilities",
    labels={"customer_action_probability": "Probability of customer action"}
)
fig.show()


**Interpretation.** The data lie in $[0,1]$, are concentrated below about $0.35$, and show a right-skewed pattern. This makes a Beta model plausible.


---
## **Part 2 — Define the Model and Domain**
### **Prompt / Problems**

Define the probability density function
$$
f(x)=
\begin{cases}
30x(1-x)^4, & 0\le x\le 1\\
0, & \text{otherwise}
\end{cases}
$$

1. State the domain.
2. Explain why the function is zero outside $[0,1]$.
3. Graph the function.
4. Identify the general shape.
---


### **Complete Solution**

The density is only active on the interval $[0,1]$ because the random variable is a probability. Outside that interval, the density must be 0.


In [ ]:
# Piecewise pdf
f_piecewise = sp.Piecewise((30*x*(1-x)**4, (x >= 0) & (x <= 1)), (0, True))
f_piecewise


In [ ]:
xs = np.linspace(-0.1, 1.1, 500)
ys = np.where((xs >= 0) & (xs <= 1), 30*xs*(1-xs)**4, 0)

fig = px.line(
    x=xs,
    y=ys,
    title="Beta(2,5) Probability Density Function",
    labels={"x": "x", "y": "f(x)"}
)
fig.update_traces(mode="lines")
fig.show()


**Shape.** The graph starts at 0, rises to a peak near $x=0.2$, then falls back to 0 at $x=1$. The density is right-skewed, meaning lower customer-action probabilities are more common than higher ones.


---
## **Part 3 — Limits and Continuity**
### **Prompt / Problems**

Compute:
$$
\lim_{x\to 0^+} f(x), \qquad \lim_{x\to 1^-} f(x)
$$

1. Compare these with the function values at the endpoints.
2. Determine continuity at $x=0$ and $x=1$.
3. State where $f$ is continuous.
---


### **Complete Solution**


In [ ]:
right_limit_0 = sp.limit(f, x, 0, dir='+')
left_limit_1 = sp.limit(f, x, 1, dir='-')
f0 = sp.simplify(f.subs(x, 0))
f1 = sp.simplify(f.subs(x, 1))

right_limit_0, left_limit_1, f0, f1


Since
$$
\lim_{x\to 0^+}f(x)=0=f(0)
\quad\text{and}\quad
\lim_{x\to 1^-}f(x)=0=f(1),
$$
the density is continuous at both endpoints. Because it is a polynomial on $(0,1)$, it is continuous on all of $[0,1]$.


---
## **Part 4 — Derivatives**
### **Prompt / Problems**

Given
$$
f(x)=30x(1-x)^4,
$$

1. Compute $f'(x)$.
2. Factor completely.
3. Find critical numbers.
4. Determine intervals of increase and decrease.
---


### **Complete Solution**


In [ ]:
fprime = sp.diff(f, x)
fprime_factored = sp.factor(fprime)
fprime, fprime_factored


In [ ]:
critical_points = sp.solve(sp.Eq(fprime_factored, 0), x)
critical_points


The derivative factors as
$$
f'(x)=30(1-x)^3(1-5x).
$$
The critical numbers from the derivative are $x=1$ and $x=\frac15$. On the open interval $(0,1)$, the interior critical number is
$$
x=\frac15.
$$

Sign analysis:
- if $0<x<\frac15$, then $1-5x>0$, so $f'(x)>0$
- if $\frac15<x<1$, then $1-5x<0$, so $f'(x)<0$

Therefore, $f$ is increasing on $(0,\frac15)$ and decreasing on $(\frac15,1)$.


---
## **Part 5 — Curve Sketching**
### **Prompt / Problems**

Use calculus to describe the graph:
- domain
- intercepts
- endpoint behavior
- increasing/decreasing intervals
- maximum point
- concavity (optional)

Sketch by hand and compare with technology.
---


### **Complete Solution**

- **Domain:** $[0,1]$ for the nonzero part of the pdf
- **Intercepts:** $f(0)=0$ and $f(1)=0$
- **Increasing:** $(0,\frac15)$
- **Decreasing:** $(\frac15,1)$
- **Maximum point:** occurs at $x=\frac15$

Now compute the exact maximum value.


In [ ]:
mode_x = sp.Rational(1, 5)
mode_y = sp.simplify(f.subs(x, mode_x))
mode_x, mode_y


So the maximum point is
$$
\left(\frac15,\,30\cdot\frac15\left(1-\frac15\right)^4\right)
=\left(\frac15,\frac{6144}{3125}\right)
\approx (0.2,1.96608).
$$

This means the density is highest around a customer-action probability of about **20%**.


---
## **Part 6 — Optimization**
### **Prompt / Problems**

1. Use $f'(x)=0$ to find the maximum of the density.
2. Solve
$$
1-5x=0.
$$
3. Identify the maximizing value of $x$.
4. Compute the maximum value of $f(x)$.
5. Interpret the result in context.
---


### **Complete Solution**


In [ ]:
solution_x = sp.solve(sp.Eq(1 - 5*x, 0), x)[0]
max_density = sp.simplify(f.subs(x, solution_x))
solution_x, max_density, float(max_density)


The density is maximized at
$$
x=\frac15=0.2.
$$
The maximum density value is
$$
f\left(\frac15\right)=\frac{6144}{3125}\approx 1.96608.
$$

Interpretation: the model places the greatest density near customers whose action probability is around **20%**.


---
## **Part 7 — Gradient Descent / Ascent on the Original PDF**
### **Prompt / Problems**

Apply gradient descent-style iteration directly to the original pdf.

Goal: approximate the value of $x$ that maximizes $f(x)$.

1. Compute
$$
f'(x)=30(1-x)^3(1-5x).
$$
2. Use the update rule
$$
x_{n+1}=x_n+\eta f'(x_n)
$$
so the iteration moves in the direction of increasing $f(x)$.
3. Choose an initial guess $x_0$ and learning rate $\eta$.
4. Perform at least 5 iterations.
5. Record a table of values.
6. Describe what value the sequence approaches.
---


### **Complete Solution**

Use gradient **ascent** here, since the goal is to maximize the density.
Choose:
- initial guess $x_0=0.6$
- learning rate $\eta=0.08$


In [ ]:
import sympy as sp
import numpy as np
#New toys!
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the function and its derivative
w = sp.Symbol('w', real=True)
# Beta(2,5) pdf on [0,1]
f = -30*w*(1-w)**4

deriv_expr = sp.diff(f, w)

# Lambdify for numerical evaluation
f_numeric = sp.lambdify(w, f, 'numpy')
df_numeric = sp.lambdify(w, deriv_expr, 'numpy')

# --- Widgets ---
initial_guess_input = widgets.FloatText(value=1.0, description='Initial Guess:')
learning_rate_input = widgets.FloatText(value=0.01, description='Learning Rate:')
start_reset_button = widgets.Button(description='Start/Reset Gradient Descent')
next_iteration_button = widgets.Button(description='Next Iteration', disabled=True)
output_widget = widgets.Output()

# --- State variables ---
current_w_val = None
current_iteration = 0

# --- Functions ---
def reset_gradient_descent(_):
    global current_w_val, current_iteration
    current_w_val = initial_guess_input.value
    current_iteration = 0
    next_iteration_button.disabled = False
    with output_widget:
        clear_output()
        print(f"Gradient Descent for f(w) = {f}")
        print(f"Derivative df/dw = {deriv_expr}")
        print(f"Initial Guess: {initial_guess_input.value}, Learning Rate: {learning_rate_input.value}")
        print("-" * 40)
    # Automatically run the first iteration after reset
    run_next_iteration(None)


def run_next_iteration(_):
    global current_w_val, current_iteration

    if current_w_val is None:
        with output_widget:
            print("Please click 'Start/Reset Gradient Descent' first.")
        return

    current_iteration += 1
    f_val = f_numeric(current_w_val)
    df_val = df_numeric(current_w_val)

    with output_widget:
        print(f"\nIteration {current_iteration}:")
        print(f"  Current w: {current_w_val:.6f}")
        print(f"  f(w): {f_val:.6f}")
        print(f"  df/dx: {df_val:.6f}")

        # Check for convergence (derivative close to zero)
        if abs(df_val) < 1e-4:
            print("\nConverged to a local extremum!")
            next_iteration_button.disabled = True
            return

        next_w = current_w_val - learning_rate_input.value * df_val
        print(f"  Next w: {next_w:.6f}")

        # Check for very small step size (also indicates convergence)
        if current_iteration > 1 and abs(next_w - current_w_val) < 1e-6:
             print("\nStep size is very small, likely converged.")
             next_iteration_button.disabled = True
             return

        current_w_val = next_w

        # Safety break for too many iterations
        if current_iteration > 500:
            print("\nMaximum iterations reached. Consider adjusting learning rate or initial guess.")
            next_iteration_button.disabled = True
            return

# --- Event Handlers ---
start_reset_button.on_click(reset_gradient_descent)
next_iteration_button.on_click(run_next_iteration)

# --- Display UI ---
ui = widgets.VBox([
    widgets.HBox([initial_guess_input, learning_rate_input]),
    widgets.HBox([start_reset_button, next_iteration_button]),
    output_widget
])

display(ui)

The iteration moves toward $x\approx 0.2$, which agrees with the calculus result for the maximizing value.


---
## **Part 8 — Riemann Sums and Definite Integral**
### **Prompt / Problems**

Approximate
$$
\int_0^1 30x(1-x)^4\,dx
$$
using:
- $L_4$
- $R_4$
- $M_4$

Then repeat with $n=8$ and compare with the exact value.
---


### **Complete Solution**


In [ ]:
f = 30*x*(1-x)**4

f_num = sp.lambdify(x, f, 'numpy')

fig = plot_riemann(f_num, 0, 1, n=4, method='R', title='Riemann rectangles for x^2')
fig.show()

In [ ]:

for method in ['L', 'R', 'M']:
    print(f"{method}4 =", riemann_sum(f_num, 0, 1, 4, method))


In [ ]:
exact_integral = sp.integrate(f, (x, 0, 1))
exact_integral


The exact value is
$$
\int_0^1 30x(1-x)^4\,dx = 1,
$$
which confirms total probability is 1.


---
## **Part 9 — Verify PDF Properties**
### **Prompt / Problems**

Show that:
1. $f(x)\ge 0$ for all $x$
2. $\int_0^1 30x(1-x)^4\,dx = 1$
---


### **Complete Solution**

For $0\le x\le 1$,
- $x\ge 0$
- $(1-x)^4\ge 0$
- $30>0$

Therefore,
$$
30x(1-x)^4\ge 0.
$$
Also,
$$
\int_0^1 30x(1-x)^4\,dx=1.
$$
So $f$ satisfies the two pdf conditions.


---
## **Part 10 — u-Substitution**
### **Prompt / Problems**

Compute
$$
P(X\le 0.4)=\int_0^{0.4}30x(1-x)^4\,dx
$$
using the substitution
$$
u=1-x,\qquad du=-dx.
$$
---


### **Complete Solution**


In [ ]:
prob_04 = sp.integrate(f, (x, 0, sp.Rational(2,5)))
prob_04_simplified = sp.nsimplify(prob_04)
prob_04_simplified, float(prob_04_simplified)


With
$$
u=1-x,\qquad x=1-u,\qquad du=-dx,
$$
we can rewrite
$$
\int 30x(1-x)^4\,dx
$$
in terms of $u$. The exact probability is
$$
P(X\le 0.4)=\frac{54432}{78125}\approx 0.69673.
$$


---
## **Part 11 — CDF**
### **Prompt / Problems**

Define
$$
F(x)=P(X\le x)
$$
by
$$
F(x)=
\begin{cases}
0, & x<0\$$4pt]
\int_0^x 30t(1-t)^4\,dt, & 0\le x\le 1\$$4pt]
1, & x>1
\end{cases}
$$

1. Simplify the middle expression.
2. Graph $F(x)$.
3. State domain and range.
4. Interpret $F(0.4)$.
---


### **Complete Solution**


In [ ]:
F_middle = sp.expand(sp.integrate(30*t*(1-t)**4, (t, 0, x)))
F_middle


In [ ]:
sp.factor(F_middle)


In [ ]:
# Graph the CDF piecewise
F_vals = np.piecewise(
    xs,
    [xs < 0, (xs >= 0) & (xs <= 1), xs > 1],
    [0, lambda z: 15*z**2 - 40*z**3 + 45*z**4 - 24*z**5 + 5*z**6, 1]
)

fig = px.line(x=xs, y=F_vals, title='CDF for Beta(2,5)', labels={'x':'x','y':'F(x)'})
fig.show()


A simplified formula for the middle piece is
$$
F(x)=15x^2-40x^3+45x^4-24x^5+5x^6,
\qquad 0\le x\le 1.
$$
So the full CDF is
$$
F(x)=
\begin{cases}
0, & x<0\[4pt]
15x^2-40x^3+45x^4-24x^5+5x^6, & 0\le x\le 1\[4pt]
1, & x>1.
\end{cases}
$$
Its domain is all real numbers and its range is $[0,1]$.

Also,
$$
F(0.4)=P(X\le 0.4)\approx 0.69673.
$$


---
## **Part 12 — Probabilities**
### **Prompt / Problems**

Compute and interpret:
1. $P(X<0.2)$
2. $P(0.2\le X\le 0.5)$
3. $P(X>0.7)$
4. one custom probability

For each, show the integral and the CDF method.
---


### **Complete Solution**


In [ ]:
F = sp.lambdify(x, F_middle, 'numpy')

p1 = sp.simplify(sp.integrate(f, (x, 0, sp.Rational(1,5))))
p2 = sp.simplify(sp.integrate(f, (x, sp.Rational(1,5), sp.Rational(1,2))))
p3 = sp.simplify(sp.integrate(f, (x, sp.Rational(7,10), 1)))
p4 = sp.simplify(sp.integrate(f, (x, sp.Rational(1,10), sp.Rational(2,5))))

probs = pd.DataFrame({
    'event': ['P(X < 0.2)', 'P(0.2 ≤ X ≤ 0.5)', 'P(X > 0.7)', 'P(0.1 ≤ X ≤ 0.4)'],
    'exact_value': [p1, p2, p3, p4],
    'decimal_value': [float(p1), float(p2), float(p3), float(p4)]
})
probs


Interpretations:
- $P(X<0.2)$ gives the probability that a customer's action probability is below 20%.
- $P(0.2\le X\le 0.5)$ gives the probability that the customer's action probability is between 20% and 50%.
- $P(X>0.7)$ gives the probability of a very high-engagement customer.
- $P(0.1\le X\le 0.4)$ is a custom marketing interval.


---
## **Part 13 — Marketing Interpretation**
### **Prompt / Problems**

Explain what the distribution suggests about customer behavior:
- where probability mass is concentrated
- likelihood of high vs low engagement
- implications for marketing strategy
---


### **Complete Solution**

The Beta$(2,5)$ density places most of its mass at lower values of $x$, with a peak near $x=0.2$. This means the model describes a population in which many customers have relatively low action probabilities, while very high action probabilities are much less common. In marketing terms, this suggests broad campaigns may reach many low-intent users and relatively few high-intent users.


---
## **Part 14 — Conclusion**
### **Prompt / Problems**

Address the following:
- Does the Beta$(2,5)$ model fit the data?
- What did derivatives reveal?
- What did integrals reveal?
- What did gradient ascent approximate?
- What would need to change to model a more engaged audience?
---


### **Complete Solution**

This proof of concept is consistent with the example dataset because the data are concentrated below about $0.35$ and show a right-skewed pattern. Derivatives revealed where the density increases, decreases, and reaches its maximum. Integrals revealed total probability and interval probabilities. Gradient ascent numerically approximated the maximizing value near $x=0.2$. To model a more engaged audience, the density would need to shift to the right so that larger customer-action probabilities become more common.
